# Top 10 Research — Colab Runner

One notebook to clone the repo and run any of the 10 problems on a GPU.

**First:** `Runtime → Change runtime type → T4 GPU` (or A100).

Workflow: edit code locally → `git push` → re-run **cell 1** here (it pulls) → run.


In [ ]:
# 1. Clone (first time) or pull (subsequent runs)
import os
REPO = 'Top10Research'
if not os.path.isdir(f'/content/{REPO}'):
    !git clone https://github.com/Sudarssan-N/Top10Research.git /content/{REPO}
else:
    !cd /content/{REPO} && git pull --ff-only
%cd /content/{REPO}
!git log --oneline -1

In [ ]:
# 2. Pick a problem + install its deps (torch is already on Colab, so we skip it)
PROBLEM = 'p01-verifier-aware-controller'  # change to p02-..., p03-..., etc.
%cd /content/Top10Research/{PROBLEM}
!pip install -q transformers datasets accelerate sympy huggingface_hub
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(no GPU — set Runtime to T4)')

## Run P1 — Phase 1 baselines (best-of-N Pareto)

Qwen2.5-1.5B is open (no HF login needed). MATH-500 + GSM8K are public.
200 examples × N=1,4,8,16 ≈ 2–4 hrs on T4, faster on A100.

In [ ]:
!python scripts/run_phase1_baselines.py \
  --config configs/phase1.yaml \
  --max-examples 200 \
  --n-values 1,4,8,16 \
  --output_dir results/phase1_real

In [ ]:
# 3. View the Pareto table + plot
import json
d = json.load(open('results/phase1_real/phase1_summary.json'))
for b, bd in d['benchmarks'].items():
    print(f'\n== {b} ({bd.get("num_examples", "?")} ex) ==')
    if 'best_of_n' in bd:
        for n, sm in bd['best_of_n']['sweep'].items():
            for s, m in sm.items():
                print(f'  N={n:>2} {s:24s} acc={m["accuracy"]:.3f} avg_out_tok={m["avg_output_tokens"]:.0f}')
from IPython.display import Image, display
try:
    display(Image('results/phase1_real/phase1_pareto.png'))
except Exception:
    pass

## (Optional) Persist results back to GitHub

Results are gitignored by default (kept off the repo). To save numbers, copy to Drive,
or download `results/phase1_real/phase1_summary.json` via the file browser.

In [ ]:
from google.colab import files
files.download('results/phase1_real/phase1_summary.json')